This mini library contains two `Python` decorate functions to aid in performance debugging by displaying the evluation times of its various subcomponents in tree-like structures.

## `@siena_decorator()`

In [1]:
from debugger import siena_debugger, flamegraph_debugger
from IPython.display import display
import ipywidgets as widgets
import numpy as np
import time
import sys

"""
Jupyter notebooks don't always behave like a terminal due to buffering, so 
simulate a better one with ipywidgets.Output.
"""
def terminal_simulator(func, *args, **kwargs):
    out = widgets.Output(layout={'border': '1px solid black'})
    display(out)
    with out:
        out.clear_output(wait=True)
        func(*args, **kwargs)
        sys.stdout.write("\033[1K\r" + " " * 50) # clear line + carriage return
        sys.stdout.flush()

Our decorator displays the wall-clock time of each line executing throughout the function call using an easy-to-interpret tree-like structure to show the structure of the stack calls. As an example, let us begin with a simple function, `simple()`.

In [2]:
@siena_debugger()
def simple(delay=1.5):
    a = 1
    b = 2
    c = a + b
    d = np.array(range(100000000))
    e = d ** 3.3
    time.sleep(delay)
    d = c * 1.0 / b
    return c

We can now run `simple()`, the result of which is a timer for each line of code. (Note that the timer isn't working in Jupyter for line 6 but it does in Terminal...). 

In [3]:
terminal_simulator(simple)

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

Our decorator is also able to handle function (and file) hierarchies. Let's create a more complicated set of nested functions, now.

In [7]:
@siena_debugger()
def complicated():
    a = 1
    nestc()
    b = 2
    c = a + b
    nesta()
    d = np.array(range(100000000))
    e = d ** 3.3
    time.sleep(0.75)
    nestb()
    d = c * 1.0 / b
    return c

def nesta():
    time.sleep(0.3)

def nestb():
    time.sleep(1)
    nesta()

def nestc():
    nestb()
    time.sleep(1.33)

We can now go ahead and run `complicated()`.

In [8]:
terminal_simulator(complicated)

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

But oh no! We're getting so many lines, it's starting to get messy! Thankfully, we can set a `threshold` parameter that only prints to screen times greater than the threshold.

In [9]:
@siena_debugger(threshold=1)
def less_complicated():
    a = 1
    nestc()
    b = 2
    c = a + b
    nesta()
    d = np.array(range(100000000))
    e = d ** 3.3
    time.sleep(0.75)
    nestb()
    d = c * 1.0 / b
    return c

def nesta():
    time.sleep(0.3)

def nestb():
    time.sleep(1)
    nesta()

def nestc():
    nestb()
    time.sleep(1.33)

In [ ]:
less_complicated()

things to add:
1) show timer functionality, use sleep to demonstrate
2) show nestin functionality
3) show how it works referencingother files but still has a scope!
4) show how i can choose a threshold